# 9 - GRU Test Set Evaluation (ROC / AUC)

Part of **RunSight** - see the repo README.

**Purpose.** Evaluates the trained GRU (aerobic vs. anaerobic classifier) without re-running training: test AUC, ROC curve, confusion matrix, a train/val/test AUC overfitting check, and an error analysis around the 0.02 label threshold. Note that the labels come from the rule-based pipeline, so the AUC measures fidelity to that rule-based label, not agreement with independent ground truth.

**Inputs**
- `artifacts/gru_model.keras` (saved by `7_gru_tuner.ipynb`)
- `artifacts/gru_arrays.npz` (test set, plus the train set for the overfitting check; written by `6_gru_preprocess.ipynb`)

**Outputs** (all in `../docs/images/`)
- `roc_auc_curve.png` (ROC curve and confusion matrix)
- `auc_train_val_test.png` (train / val / test AUC bars)
- `error_by_threshold_distance.png` (error rate by distance to the 0.02 label threshold)
- Console output: model summary, classification report, AUCs, error tables

# GRU Model - Test Set Evaluation (ROC / AUC)

**What does this notebook do?**
`7_gru_tuner.ipynb` trained the model and saved it as `gru_model.keras`. This notebook reads that saved model and the test set in `gru_arrays.npz` from disk and produces the test AUC, the ROC curve and the confusion matrix. **There is no need to re-run training** (the tuner search took ~5 hours); this notebook finishes in a few minutes.

**Why a separate notebook?**
Re-running a 5-hour search and training just to produce the evaluation figures (presentation, README) is unnecessary. Since the model and the test set are stored on disk, evaluation can be done independently of training. Only `X_test_scaled` and `y_test` are loaded here; the training set is not loaded into memory.

**Important note - what does this AUC measure?**
The `y_test` label is the `training_zone` label produced by the rule-based pipeline (`run_zone_pipeline`); it is not independent ground truth. So the AUC measures the GRU's **fidelity** to the rule-based label.

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, auc as sk_auc, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

BASE = 'artifacts'
IMG_DIR = '../docs/images'
os.makedirs(IMG_DIR, exist_ok=True)

## 1. Load the test set and the saved model

`np.load` opens the `.npz` file lazily: only the requested keys are read into memory, not the whole 556 MB file.

In [ ]:
data = np.load(f'{BASE}/gru_arrays.npz')
X_test_scaled = data['X_test_scaled']
y_test = data['y_test']
print('X_test_scaled:', X_test_scaled.shape, '| y_test:', y_test.shape, '| class counts:', np.bincount(y_test))

model = tf.keras.models.load_model(f'{BASE}/gru_model.keras')
model.summary()

## 2. Prediction and test AUC

For each run the model outputs a probability between 0 and 1; the ROC curve shows how this probability behaves across all thresholds.

In [ ]:
y_pred_proba = model.predict(X_test_scaled, batch_size=256).ravel()
y_pred = (y_pred_proba >= 0.5).astype(int)

fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
test_auc = sk_auc(fpr, tpr)
print(f'test AUC: {test_auc:.4f}')
print(classification_report(y_test, y_pred, target_names=['aerobic', 'anaerobic']))

## 3. ROC curve and confusion matrix

**How to read the ROC curve?** The dashed diagonal is the curve of a model that guesses randomly (AUC = 0.5). The higher above the diagonal (toward the top left) the model's curve lies, the stronger its ability to separate the classes. The black dot is the point where the model actually operates at the 0.5 threshold.

In [ ]:
op_idx = np.argmin(np.abs(thresholds - 0.5))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.plot(fpr, tpr, color='#f26b21', lw=2, label=f'GRU (AUC = {test_auc:.3f})')
ax.plot([0, 1], [0, 1], '--', color='gray', lw=1, label='Random (AUC = 0.500)')
ax.scatter(fpr[op_idx], tpr[op_idx], color='black', zorder=5, label='Threshold = 0.5')
ax.set_xlabel('False positive rate (1 - specificity)')
ax.set_ylabel('True positive rate (sensitivity)')
ax.set_title('ROC curve - test set')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred), display_labels=['aerobic', 'anaerobic']
).plot(ax=axes[1], colorbar=False, cmap='Oranges')
axes[1].set_title('Confusion matrix (threshold = 0.5)')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/roc_auc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Overfitting check: train / val / test AUC

`7_gru_tuner.ipynb` used `validation_split=0.2` during training: the **first 80%** of the training set is the data the model saw (fit), and the **last 20%** is validation (val). The same split is repeated here and the saved model's AUC on the three subsets is compared.

**How to read it?** If train AUC >> val/test AUC, the model has memorized. If the three values are close to each other, there is no overfitting. Note: the saved model carries the weights of the best val_auc epoch via `restore_best_weights=True`, so the val AUC may be slightly optimistic; the test AUC is the truly reliable measure.

In [ ]:
from sklearn.metrics import roc_auc_score

data = np.load(f'{BASE}/gru_arrays.npz')
X_tr_all, y_tr_all = data['X_train_scaled'], data['y_train']
n_fit = int(len(X_tr_all) * 0.8)          # keras validation_split=0.2 -> last 20% is val
X_fit, y_fit = X_tr_all[:n_fit], y_tr_all[:n_fit]
X_val, y_val = X_tr_all[n_fit:], y_tr_all[n_fit:]

p_fit = model.predict(X_fit, batch_size=256).ravel()
p_val = model.predict(X_val, batch_size=256).ravel()
auc_fit, auc_val = roc_auc_score(y_fit, p_fit), roc_auc_score(y_val, p_val)
print(f'train AUC: {auc_fit:.4f} | val AUC: {auc_val:.4f} | test AUC: {test_auc:.4f}')

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['train', 'val', 'test'], [auc_fit, auc_val, test_auc], color=['#4c78a8', '#f2a03d', '#f26b21'])
for b in bars:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.005, f'{b.get_height():.3f}', ha='center')
ax.set_ylim(0.5, 1.0); ax.set_ylabel('AUC'); ax.set_title('Train / val / test AUC')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/auc_train_val_test.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Error analysis: are misclassifications concentrated near the label threshold (0.02)?

The claim made in the presentation: *"Misclassifications are concentrated in runs close to the 0.02 boundary value in the labeling rule."* This cell tests that claim.

The label was produced with `y_test = (yc_test > 0.02)` (`yc_test` = fraction of time above the threshold). If the claim is true, the error rate for runs whose `yc_test` is close to 0.02 should be clearly higher than for those far from it.

**How to read it?** Two measures: (1) error rate inside the band vs. error rate outside the band; (2) what percentage of all errors fall in the band vs. what percentage of all samples fall in the band. If the error share is clearly larger than the sample share, the claim is supported. If the result is the opposite of what was expected, slide 13 and the Q&A sentence must be changed.

In [ ]:
import pandas as pd
yc_test = np.load(f'{BASE}/gru_arrays.npz')['yc_test']
err = (y_pred != y_test)
print('overall error rate:', err.mean().round(4), '| number of errors:', err.sum())
print('number of samples with yc_test exactly equal to 0.02:', int((yc_test == 0.02).sum()))

# bins over yc_test (fine around 0.02, coarser farther away)
edges = [-1e-9, 0.0, 0.01, 0.02, 0.03, 0.05, 0.10, 0.30, 0.60, 1.0]
labels = ['=0', '(0,0.01]', '(0.01,0.02]', '(0.02,0.03]', '(0.03,0.05]',
          '(0.05,0.10]', '(0.10,0.30]', '(0.30,0.60]', '(0.60,1.0]']
bins = pd.cut(yc_test, bins=edges, labels=labels)
tab = (pd.DataFrame({'bin': bins, 'err': err})
       .groupby('bin', observed=True)['err'].agg(n='size', error_rate='mean', n_errors='sum'))
print(tab.round(3))

# band around 0.02: |yc - 0.02| <= 0.03
band = np.abs(yc_test - 0.02) <= 0.03
print(f"\nband (|yc-0.02|<=0.03): sample share {band.mean():.1%} | error share {err[band].sum() / err.sum():.1%}")
print(f"error rate: inside band {err[band].mean():.1%} | outside band {err[~band].mean():.1%}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(tab.index.astype(str), tab['error_rate'], color='#f26b21')
ax.axhline(err.mean(), ls='--', color='gray', label=f'overall error rate ({err.mean():.2f})')
ax.set_ylabel('Error rate'); ax.set_xlabel('yc_test (fraction of time above threshold)')
ax.set_title('Error rate by proximity to the label threshold (0.02)'); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=30); plt.tight_layout()
plt.savefig(f'{IMG_DIR}/error_by_threshold_distance.png', dpi=150, bbox_inches='tight')
plt.show()